# __Main_part__

__Import libraries__

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import os
import shutil
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import cv2
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import torch.optim as optim
import torchvision.models as models
print(torch.cuda.is_available())

False


__Task_1 - Preprocessing__

Распаковка

In [ ]:
#!cd "drive/MyDrive/Colab_Notebooks/s21_ML9/data/" && unzip -q Images.zip && rm -rf Images.zip
#!rm -rf "drive/MyDrive/Colab_Notebooks/s21_ML9/data/__MACOSX"

Удаление тестовых данных

In [ ]:
# !ls -1 "drive/MyDrive/Colab_Notebooks/s21_ML9/data/Images/" | wc -l

In [ ]:
#valid_ids = set(df["img_IDS"].values)

#all_files = os.listdir(images_dir)
#to_remove = [f for f in all_files if os.path.splitext(f)[0] not in valid_ids]

#print(f"Всего файлов в папке: {len(all_files)}")
#print(f"В датасете: {len(valid_ids)}")
#print(f"Лишних файлов: {len(to_remove)}")

In [ ]:
#for f in to_remove:
#    os.remove(os.path.join(images_dir, f))

In [ ]:
#!ls -1 "drive/MyDrive/Colab_Notebooks/s21_ML9/data/Images/" | wc -l

In [3]:
df_train = pd.read_csv("drive/MyDrive/Colab_Notebooks/s21_ML9/data/train.csv")
print(df_train.shape)
df_train.head(3)

(4186, 2)


,img_IDS,Label
0,ImageID_OL30LJ9I,You
1,ImageID_UQCX3D3Q,Love
2,ImageID_U5GCXHVF,Mosque


In [4]:
df_val = pd.read_csv("drive/MyDrive/Colab_Notebooks/s21_ML9/data/val.csv")
print(df_val.shape)

(2063, 2)


Проверка на пропуски и дубли

In [ ]:
df_train["Label"].isna().sum()

np.int64(0)

In [ ]:
print("Count of duplicated images:", df_train["img_IDS"].duplicated().sum())

Count of duplicated images: 0


Проверка на баланс классов

In [ ]:
df_train["Label"].unique()

array(['You', 'Love', 'Mosque', 'Enough/Satisfied', 'Church', 'Friend',
       'Temple', 'Seat', 'Me'], dtype=object)

In [ ]:
df_train["Label"].value_counts()

,count
Label,
Mosque,466
You,465
Love,465
Enough/Satisfied,465
Church,465
Friend,465
Temple,465
Seat,465
Me,465


Сплиттинг и сохранение файлов в .csv

In [ ]:
#train_dir = os.path.join(images_dir, "images/train")
#val_dir = os.path.join(images_dir, "images/val")

#os.makedirs(train_dir, exist_ok=True)
#os.makedirs(val_dir, exist_ok=True)

In [ ]:
#X = df.drop(columns=["Label"])
#y = df["Label"]

#X_train, X_val, y_train, y_val = train_test_split(X, y,
#                                                  test_size=0.33,
#                                                  stratify=y,
#                                                  random_state=42)

#print(f"Train data shape: {int((X_train.shape[0] / df.shape[0]) * 100)}%" )
#print(f"Val data shape: {int((X_val.shape[0] / df.shape[0]) * 100)}%" )

In [ ]:
#df_train = pd.concat([X_train, y_train], axis=1)
#df_val = pd.concat([X_val, y_val], axis=1)

#df_train.to_csv(os.path.join(images_dir, "train.csv"), index=False)
#df_val.to_csv(os.path.join(images_dir, "val.csv"), index=False)

In [ ]:
#def move_files(df_split, target_dir):
#  for img_id in df_split["img_IDS"]:
#    for ext in [".jpg"]:
#      src = os.path.join(images_dir, img_id + ext)
#      if os.path.exists(src):
#        shutil.copy(src, os.path.join(target_dir, img_id + ext))
#        break

#move_files(df_train, train_dir)
#move_files(df_val, val_dir)

Лейбл энкодер

In [ ]:
#le = LabelEncoder()
#df_train["LabelEncoded"] = le.fit_transform(df_train["Label"])
#df_val["LabelEncoded"] = le.fit_transform(df_val["Label"])

In [ ]:
#df_train.drop(['Label'], axis=1, inplace=True)
#df_val.drop(['Label'], axis=1, inplace=True)

#df_train.rename(columns={"LabelEncoded" : "Label"}, inplace=True)
#df_val.rename(columns={"LabelEncoded" : "Label"}, inplace=True)

In [ ]:
#df_train.to_csv("drive/MyDrive/Colab_Notebooks/s21_ML9/data/train_encoded.csv")
#df_val.to_csv("drive/MyDrive/Colab_Notebooks/s21_ML9/data/val_encoded.csv")

data/  \
├── train_encoded.csv \
├── val_encoded.csv  \
└── images/  \
....├── train/ (4186 files)  \
....└── val/   (2063 files)

__Task_2 - Implementation class for creating Dataloader__

In [5]:
class CustomImageDataset(Dataset):
  def __init__(self, csv_file, img_dir, transform=None):
    self.data = pd.read_csv(csv_file)
    self.img_dir = img_dir
    self.transform = transform if transform else transforms.ToTensor()

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    img_name = str(self.data.loc[idx, "img_IDS"])
    img_path = os.path.join(self.img_dir, img_name + ".jpg")

    image = cv2.imread(img_path)
    if image is None:
      raise FileNotFoundError(f"File not found: {img_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    label = self.data.loc[idx, "Label"]
    label = torch.tensor(label, dtype=torch.long)

    if self.transform:
      image = self.transform(image)

    return image, label

In [6]:
transform = transforms.Compose([
  transforms.ToPILImage(),
  transforms.Resize((32, 32)),
  transforms.ToTensor(),
])

train_dataset = CustomImageDataset(
  csv_file="drive/MyDrive/Colab_Notebooks/s21_ML9/data/train_encoded.csv",
  img_dir="drive/MyDrive/Colab_Notebooks/s21_ML9/data/images/train/",
  transform=transform
)

val_dataset = CustomImageDataset(
  csv_file="drive/MyDrive/Colab_Notebooks/s21_ML9/data/val_encoded.csv",
  img_dir="drive/MyDrive/Colab_Notebooks/s21_ML9/data/images/val/",
  transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

In [ ]:
print(train_dataset, train_dataset.data.head(), len(train_dataset), sep="\n\n")


   Unnamed: 0           img_IDS  Label
0           0  ImageID_OL30LJ9I      8
1           1  ImageID_UQCX3D3Q      3
2           2  ImageID_U5GCXHVF      5
3           3  ImageID_UDN3D0DK      1
4           4  ImageID_GUMIW01H      0

4186


In [ ]:
print(val_dataset, val_dataset.data.head(), len(val_dataset), sep="\n\n")


   Unnamed: 0           img_IDS  Label
0           0  ImageID_3HL2GAMR      6
1           1  ImageID_Z26SF2VE      1
2           2  ImageID_35JAUO9I      6
3           3  ImageID_1WK1L5QZ      1
4           4  ImageID_NI20U25H      3

2063


__Task_3 - LeNet architecture__

In [ ]:
class LeNet(nn.Module):
  def __init__(self, num_classes=2):
    super(LeNet, self).__init__()

    self.conv1 = nn.Conv2d(3, 6, kernel_size=5)
    self.pool = nn.MaxPool2d(2, 2)
    self.conv2 = nn.Conv2d(6, 16, kernel_size=5)

    with torch.no_grad():
      x = torch.zeros(1, 3, 32, 32)
      x = self.pool(F.relu(self.conv1(x)))
      x = self.pool(F.relu(self.conv2(x)))
      n_features = x.view(1, -1).size(1)

    self.fc1 = nn.Linear(n_features, 120)
    self.fc2 = nn.Linear(120, 84)
    self.fc3 = nn.Linear(84, num_classes)

  def forward(self, x):
    x = F.relu(self.conv1(x))
    x = self.pool(x)
    x = F.relu(self.conv2(x))
    x = self.pool(x)
    x = torch.flatten(x, 1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

__Task_4 - Pipeline for training, validation, checking metrics__

In [7]:
def train_epoch(model, loader, optimizer, criterion, device, num_classes, use_mixup=False, use_cutmix=False):
  model.train()
  total_loss = 0
  for data, target in loader:
    data, target = data.to(device), target.to(device)
    target_one_hot = F.one_hot(target, num_classes=num_classes).float()

    if use_mixup: data, targets_a, targets_b, lam = mixup_data(data, target_one_hot)
    elif use_cutmix: data, targets_a, targets_b, lam = cutmix_data(data, target_one_hot)
    else: targets_a, targets_b, lam = target_one_hot, target_one_hot, 1.0

    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target_one_hot)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
  return total_loss / len(loader)

In [8]:
def validate(model, loader, criterion, device, num_classes):
  model.eval()
  val_loss = 0
  all_targets, all_outputs = [], []
  with torch.no_grad():
    for data, target in loader:
      data, target = data.to(device), target.to(device)
      target_one_hot = F.one_hot(target, num_classes=num_classes).float()

      output = model(data)
      loss = criterion(output, target_one_hot)
      val_loss += loss.item()

      probs = torch.sigmoid(output).cpu().numpy()
      all_outputs.append(probs)
      all_targets.append(target_one_hot.cpu().numpy())

  all_outputs = np.vstack(all_outputs)
  all_targets = np.vstack(all_targets)

  roc_auc = roc_auc_score(all_targets, all_outputs, multi_class="ovr")
  return val_loss / len(loader), roc_auc

In [ ]:
def main():
  num_classes = 9
  n_epochs = 20
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model = LeNet(num_classes=9).to(device)
  optimizer = optim.Adam(model.parameters(), lr=1e-3)
  criterion = nn.BCEWithLogitsLoss()

  for epoch in range(n_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device, num_classes=num_classes)
    val_loss, roc_auc = validate(model, val_loader, criterion, device, num_classes=num_classes)
    print(f"Epoch {epoch+1:02d}/{n_epochs}:    Train_loss={train_loss:.4f}    Val_loss={val_loss:.4f}    ROC_AUC={roc_auc:.4f}")

if __name__ == "__main__":
  main()

Epoch 01/20:    Train_loss=0.3832    Val_loss=0.3505    ROC_AUC=0.5018
Epoch 02/20:    Train_loss=0.3507    Val_loss=0.3509    ROC_AUC=0.5053
Epoch 03/20:    Train_loss=0.3504    Val_loss=0.3494    ROC_AUC=0.5121
Epoch 04/20:    Train_loss=0.3502    Val_loss=0.3492    ROC_AUC=0.5276
Epoch 05/20:    Train_loss=0.3490    Val_loss=0.3474    ROC_AUC=0.5697
Epoch 06/20:    Train_loss=0.3457    Val_loss=0.3414    ROC_AUC=0.6276
Epoch 07/20:    Train_loss=0.3378    Val_loss=0.3339    ROC_AUC=0.6827
Epoch 08/20:    Train_loss=0.3231    Val_loss=0.3173    ROC_AUC=0.7312
Epoch 09/20:    Train_loss=0.3029    Val_loss=0.2995    ROC_AUC=0.7745
Epoch 10/20:    Train_loss=0.2847    Val_loss=0.2871    ROC_AUC=0.7921
Epoch 11/20:    Train_loss=0.2699    Val_loss=0.2854    ROC_AUC=0.8071
Epoch 12/20:    Train_loss=0.2613    Val_loss=0.2810    ROC_AUC=0.8163
Epoch 13/20:    Train_loss=0.2495    Val_loss=0.2709    ROC_AUC=0.8277
Epoch 14/20:    Train_loss=0.2409    Val_loss=0.2643    ROC_AUC=0.8380
Epoch 

__Task_5 - Resnet18 architecture__

In [21]:
train_transform = transforms.Compose([
  transforms.ToPILImage(),
  transforms.Resize((224, 224)),
  transforms.ToTensor(),
])

train_dataset = CustomImageDataset(
  csv_file="drive/MyDrive/Colab_Notebooks/s21_ML9/data/train_encoded.csv",
  img_dir="drive/MyDrive/Colab_Notebooks/s21_ML9/data/images/train/",
  transform=transform
)

val_dataset = CustomImageDataset(
  csv_file="drive/MyDrive/Colab_Notebooks/s21_ML9/data/val_encoded.csv",
  img_dir="drive/MyDrive/Colab_Notebooks/s21_ML9/data/images/val/",
  transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
use_amp = True

In [ ]:
def main():
  num_classes = 9
  n_epochs = 4
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  model = models.resnet18(pretrained=True)
  model.fc = nn.Linear(model.fc.in_features, num_classes)
  model = model.to(device)

  optimizer = optim.Adam(model.parameters(), lr=1e-3)
  criterion = nn.BCEWithLogitsLoss()

  for epoch in range(n_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device, num_classes)
    val_loss, roc_auc = validate(model, val_loader, criterion, device, num_classes)
    print(f"Epoch {epoch+1}:    Train_loss={train_loss:.4f}    Val_loss={val_loss:.4f}    ROC_AUC={roc_auc:.4f}")

if __name__ == "__main__":
  main()

Epoch 1:    Train_loss=0.3136    Val_loss=0.2641    ROC_AUC=0.8675
Epoch 2:    Train_loss=0.2289    Val_loss=0.2245    ROC_AUC=0.8935
Epoch 3:    Train_loss=0.1977    Val_loss=0.2208    ROC_AUC=0.9045
Epoch 4:    Train_loss=0.1779    Val_loss=0.2107    ROC_AUC=0.9126


__Task_6 - Augmentations__

In [22]:
train_transform = transforms.Compose([
  transforms.ToPILImage(),
  transforms.Resize((224, 224)),
  transforms.RandomHorizontalFlip(p=0.5),
  transforms.RandomVerticalFlip(p=0.2),
  transforms.RandomRotation(degrees=15),
  transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
  transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
  transforms.ToPILImage(),
  transforms.Resize((224, 224)),
  transforms.ToTensor(),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = CustomImageDataset(
  csv_file="drive/MyDrive/Colab_Notebooks/s21_ML9/data/train_encoded.csv",
  img_dir="drive/MyDrive/Colab_Notebooks/s21_ML9/data/images/train/",
  transform=train_transform
)

val_dataset = CustomImageDataset(
  csv_file="drive/MyDrive/Colab_Notebooks/s21_ML9/data/val_encoded.csv",
  img_dir="drive/MyDrive/Colab_Notebooks/s21_ML9/data/images/val/",
  transform=val_transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
use_amp = True

In [23]:
def main():
  num_classes = 9
  n_epochs = 4
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  model = models.resnet18(pretrained=True)
  model.fc = nn.Linear(model.fc.in_features, num_classes)
  model = model.to(device)

  optimizer = optim.Adam(model.parameters(), lr=1e-3)
  criterion = nn.BCEWithLogitsLoss()

  for epoch in range(n_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device, num_classes)
    val_loss, roc_auc = validate(model, val_loader, criterion, device, num_classes)
    print(f"Epoch {epoch+1}:    Train_loss={train_loss:.4f}    Val_loss={val_loss:.4f}    ROC_AUC={roc_auc:.4f}")

if __name__ == "__main__":
  main()

Epoch 1:    Train_loss=0.2154    Val_loss=0.1380    ROC_AUC=0.9651
Epoch 2:    Train_loss=0.1352    Val_loss=0.0895    ROC_AUC=0.9826
Epoch 3:    Train_loss=0.1171    Val_loss=0.0945    ROC_AUC=0.9839
Epoch 4:    Train_loss=0.1051    Val_loss=0.1033    ROC_AUC=0.9793


__Task_7 - MixUp and CutMix__

In [10]:
def mixup_data(x, y, alpha=0.4):
  if alpha > 0: lam = np.random.beta(alpha, alpha)
  else: lam = 1

  batch_size = x.size()[0]
  index = torch.randperm(batch_size).to(x.device)

  mixed_x = lam * x + (1 - lam) * x[index, :]
  y_a, y_b = y, y[index]
  return mixed_x, y_a, y_b, lam

In [11]:
def cutmix_data(x, y, alpha=1.0):
  if alpha > 0: lam = np.random.beta(alpha, alpha)
  else: lam = 1
  batch_size, _, H, W = x.size()
  index = torch.randperm(batch_size).to(x.device)

  cut_rat = np.sqrt(1. - lam)
  cut_w = int(W * cut_rat)
  cut_h = int(H * cut_rat)

  cx = np.random.randint(W)
  cy = np.random.randint(H)

  x1 = np.clip(cx - cut_w // 2, 0, W)
  y1 = np.clip(cy - cut_h // 2, 0, H)
  x2 = np.clip(cx + cut_w // 2, 0, W)
  y2 = np.clip(cy + cut_h // 2, 0, H)

  x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
  y_a, y_b = y, y[index]
  lam = 1 - ((x2 - x1) * (y2 - y1) / (H * W))
  return x, y_a, y_b, lam

In [17]:
def main():
  num_classes = 9
  n_epochs = 10
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  model = models.resnet18(pretrained=True)
  model.fc = nn.Linear(model.fc.in_features, num_classes)
  model = model.to(device)

  optimizer = optim.Adam(model.parameters(), lr=1e-3)
  criterion = nn.BCEWithLogitsLoss()

  for epoch in range(n_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device, num_classes, use_mixup=True, use_cutmix=False)
    val_loss, roc_auc = validate(model, val_loader, criterion, device, num_classes)
    print(f"Epoch {epoch+1:02d}/{n_epochs}:    Train_loss={train_loss:.4f}    Val_loss={val_loss:.4f}    ROC_AUC={roc_auc:.4f}")

if __name__ == "__main__":
  main()

Epoch 01/10:    Train_loss=0.3762    Val_loss=0.3379    ROC_AUC=0.6929
Epoch 02/10:    Train_loss=0.3488    Val_loss=0.4845    ROC_AUC=0.5928
Epoch 03/10:    Train_loss=0.3468    Val_loss=0.3224    ROC_AUC=0.7276
Epoch 04/10:    Train_loss=0.3483    Val_loss=0.3278    ROC_AUC=0.7287
Epoch 05/10:    Train_loss=0.3359    Val_loss=0.3218    ROC_AUC=0.7815
Epoch 06/10:    Train_loss=0.3283    Val_loss=0.9051    ROC_AUC=0.5589
Epoch 07/10:    Train_loss=0.3322    Val_loss=0.3440    ROC_AUC=0.7822
Epoch 08/10:    Train_loss=0.3343    Val_loss=0.3665    ROC_AUC=0.7884
Epoch 09/10:    Train_loss=0.3212    Val_loss=0.2776    ROC_AUC=0.8559
Epoch 10/10:    Train_loss=0.3360    Val_loss=0.2898    ROC_AUC=0.8459


In [18]:
def main():
  num_classes = 9
  n_epochs = 10
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  model = models.resnet18(pretrained=True)
  model.fc = nn.Linear(model.fc.in_features, num_classes)
  model = model.to(device)

  optimizer = optim.Adam(model.parameters(), lr=1e-3)
  criterion = nn.BCEWithLogitsLoss()

  for epoch in range(n_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device, num_classes, use_mixup=False, use_cutmix=True)
    val_loss, roc_auc = validate(model, val_loader, criterion, device, num_classes)
    print(f"Epoch {epoch+1:02d}/{n_epochs}:    Train_loss={train_loss:.4f}    Val_loss={val_loss:.4f}    ROC_AUC={roc_auc:.4f}")

if __name__ == "__main__":
  main()

Epoch 01/10:    Train_loss=0.3703    Val_loss=0.3416    ROC_AUC=0.7168
Epoch 02/10:    Train_loss=0.3370    Val_loss=0.4184    ROC_AUC=0.6562
Epoch 03/10:    Train_loss=0.3255    Val_loss=0.2900    ROC_AUC=0.8013
Epoch 04/10:    Train_loss=0.3207    Val_loss=0.2923    ROC_AUC=0.7986
Epoch 05/10:    Train_loss=0.3143    Val_loss=0.2584    ROC_AUC=0.8595
Epoch 06/10:    Train_loss=0.3036    Val_loss=0.2831    ROC_AUC=0.8293
Epoch 07/10:    Train_loss=0.3010    Val_loss=0.2569    ROC_AUC=0.8609
Epoch 08/10:    Train_loss=0.2973    Val_loss=0.2561    ROC_AUC=0.8686
Epoch 09/10:    Train_loss=0.2980    Val_loss=0.2411    ROC_AUC=0.8796
Epoch 10/10:    Train_loss=0.2905    Val_loss=0.2537    ROC_AUC=0.8725


# __Bonus_part__